In [2]:
code = 'SRE_DAY_LOW_PSL'
pickle_path = 'C:/PICKLE/'
parameter_path = f'Parameter_{code}.csv'
meta_data_path = f"Parameter_{code}_MetaData.csv"
output_csv_path = f'{code}_output\\'

import sys
sys.path.append(pickle_path)  # Add the directory containing the module
from pgcbacktest.BtParameters import *
from pgcbacktest.BacktestOptions import *

try:
    parameter, parameter_len = get_parameter_data(code, parameter_path)
    meta_data, meta_row_nos = get_meta_data(code, meta_data_path)
    os.makedirs(output_csv_path, exist_ok=True)
except Exception as e:
    input(str(e))

In [3]:
def SRE_per_minute_mtm(bt, start_time, end_time, orderside, sl, intra_sl, om, re_entries, up_buffer, seperate=False):
    try:
        start_dt = datetime.datetime.combine(bt.current_date, start_time)
        end_dt = datetime.datetime.combine(bt.current_date, end_time)

        low_break, low_break_time = bt.day_low_break(start_dt, end_dt, up_buffer)
        if low_break:
            start_dt = low_break_time
        else :
            return None
        ce_scrip, pe_scrip, ce_price, pe_price, future_price, start_dt = bt.get_strike(start_dt, end_dt, om=om)
        if ce_scrip is None: return None

        entry_time = start_dt
        std_sl_time, std_mtm_data = bt.sl_check_combine_leg(start_dt, end_dt, ce_scrip, pe_scrip, sl=sl, intra_sl=intra_sl, orderside=orderside, per_minute_mtm=True)
        
        time_index = get_pm_time_index(bt.current_date)
        std_mtm_data0 = set_pm_time_index(std_mtm_data, time_index)
        re_std_mtm_data = set_pm_time_index(pd.Series(), time_index)
        
        for re_no in range(max_re):
            
            if std_sl_time and re_no < re_entries and (std_sl_time < end_dt - datetime.timedelta(minutes=5)):
                start_dt = std_sl_time
                ce_scrip, pe_scrip, ce_price, pe_price, _, start_dt = bt.get_strike(start_dt, end_dt, om=om)
                
                if ce_scrip is None:
                    std_sl_time = ''
                    continue

                std_sl_time, std_mtm_data = bt.sl_check_combine_leg(start_dt, end_dt, ce_scrip, pe_scrip, sl=sl, intra_sl=intra_sl, orderside=orderside, per_minute_mtm=True)
                std_mtm_data = set_pm_time_index(std_mtm_data, time_index)
                re_std_mtm_data += std_mtm_data
            else:
                break
        
        if seperate:
            return std_mtm_data0, re_std_mtm_data
        else:
            return std_mtm_data0 + re_std_mtm_data

    except Exception as e:
        print(e, [bt.index, bt.current_date, start_time, end_time, orderside, sl, intra_sl, om, re_entries])
        return

In [4]:
def SRE_PSL(bt, start_time, end_time, last_trade_time, trade_interval, orderside, sl, intra_sl, om, re_entries, up_buffer):

    try:
        start_dt = datetime.datetime.combine(bt.current_date, start_time)
        end_dt = datetime.datetime.combine(bt.current_date, end_time)
        last_trade_dt = datetime.datetime.combine(bt.current_date, last_trade_time)
        
        time_range = pd.date_range(start_dt, last_trade_dt, freq=trade_interval.lower())
        time_index = get_pm_time_index(bt.current_date)
        
        entry_time = start_dt
        per_minute_mtm = set_pm_time_index(pd.Series(), time_index)
        for re_time in time_range:
            re_time = re_time.time()
            per_minute_trade = SRE_per_minute_mtm(bt, re_time, end_time, orderside, sl, intra_sl, om, re_entries, up_buffer)
            if per_minute_trade is not None:
                per_minute_mtm += per_minute_trade

        notinal_value = 12
        fund = 100_00_00_00
        total_minutes = len(time_range)
        future_price = bt.future_data['close'].iloc[0]
        margin_per_share = future_price * (notinal_value / 100)
        per_minute_margin = int(fund/total_minutes)
        shares_per_minute = int(per_minute_margin/margin_per_share)
        per_minute_mtm_total = per_minute_mtm * shares_per_minute
        
        mtm_dict = {}
        for mtm_percent in mtm_percent_stop:
            check_mtm = fund * mtm_percent/100

            try:
                if check_mtm > 0:
                    time = per_minute_mtm_total[per_minute_mtm_total > check_mtm].index[0]
                elif check_mtm < 0:
                    time = per_minute_mtm_total[per_minute_mtm_total < check_mtm].index[0]

                mtm = per_minute_mtm_total.loc[time + datetime.timedelta(minutes=1)]
            except:
                time, mtm = '', per_minute_mtm_total.iloc[-1]

            mtm_dict[mtm_percent] = (time, mtm)
            
        mtm_time_list = [item for value in mtm_dict.values() for item in value]
        
        return [code, bt.index, start_time, end_time, last_trade_time, trade_interval, orderside, sl, intra_sl, om, re_entries, bt.current_date.date(), bt.current_date.day_name(), bt.dte, entry_time.time(), future_price, fund] + mtm_time_list
    except Exception as e:
        print(e, [bt.index, bt.current_date, start_time, end_time, last_trade_time, trade_interval, orderside, sl, intra_sl, om, re_entries])
        return

In [5]:
for row_idx in range(len(meta_data)):
    
    if row_idx in meta_row_nos and meta_data.loc[row_idx, 'run']:
        try:
            meta_row = meta_data.iloc[row_idx]
            index, dte, from_date, to_date, start_time, end_time, date_lists = get_meta_row_data(meta_row, pickle_path)
            max_re = 7
            
            mtm_percent_stop = [-0.1, -0.2, -0.3, -0.4, -0.5, -0.6, -0.7, -0.8, -0.9, -1, -1.25, -1.5, -1.75, -2, -2.5, -3, -3.5, -4, -10, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1, 1.25, 1.5, 1.75, 2, 2.5, 3, 3.5, 4]
            log_cols =('P_Strategy/P_Index/P_StartTime/P_EndTime/P_LastTradeTime/P_TradeInterval/P_OrderSide/P_SL/P_intraSL/P_OM/P_ReEntries/Date/Day/DTE/EntryTime/Future/Fund')
            for mtmp in mtm_percent_stop:
                log_cols += f'/{mtmp:.2f}.Time/{mtmp:.2f}.PNL'
            log_cols = log_cols.split('/')

            for current_date in date_lists:

                file_name = f"{index} {current_date.date()} {code}"
                if not is_file_exists(output_csv_path, file_name, parameter_len):

                    t1 = datetime.datetime.now()
                    print(f"Row-{row_idx} | File-{file_name} | Total-{parameter_len}")
                    
                    bt = IntradayBacktest(pickle_path, index, current_date, dte, start_time, end_time)
                    
                    for idx, i in enumerate(range(0, parameter_len, chunk_size), start=1):
                        chunck_file_name = f"{output_csv_path}{file_name} No-{idx}.parquet"
                        print(chunck_file_name)
                        
                        chunk_parameter = parameter.iloc[i:i+chunk_size]
                        chunk = [SRE_PSL(bt, row['entry_time'], row['exit_time'], row['last_trade_time'], row['trade_interval'], row['orderside'], row['sl'], row['intra_sl'], row['om'], row['re_entries'], row['up_buffer']) for idx, row in tqdm(chunk_parameter.iterrows(), total=len(chunk_parameter), colour='GREEN')]
                        save_chunk_data(chunk, log_cols, chunck_file_name)
                        
                        del chunk
                        del chunk_parameter
                        gc.collect()

                    del bt
                    gc.collect()
                    
                    t2 = datetime.datetime.now()
                    print(t2-t1)
        except Exception as e:
            input(str(e))

Row-0 | File-NIFTY 2022-01-06 SRE_DAY_LOW_PSL | Total-1
SRE_DAY_LOW_PSL_output\NIFTY 2022-01-06 SRE_DAY_LOW_PSL No-1.parquet


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  5.32it/s]


0:00:00.456740
Row-0 | File-NIFTY 2022-01-13 SRE_DAY_LOW_PSL | Total-1
SRE_DAY_LOW_PSL_output\NIFTY 2022-01-13 SRE_DAY_LOW_PSL No-1.parquet


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  6.80it/s]


0:00:00.344254
Row-0 | File-NIFTY 2022-01-20 SRE_DAY_LOW_PSL | Total-1
SRE_DAY_LOW_PSL_output\NIFTY 2022-01-20 SRE_DAY_LOW_PSL No-1.parquet


...

100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  6.15it/s]


0:00:00.373853
Row-0 | File-NIFTY 2022-04-07 SRE_DAY_LOW_PSL | Total-1
SRE_DAY_LOW_PSL_output\NIFTY 2022-04-07 SRE_DAY_LOW_PSL No-1.parquet


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  5.86it/s]


0:00:00.402288
Row-0 | File-NIFTY 2022-04-13 SRE_DAY_LOW_PSL | Total-1
SRE_DAY_LOW_PSL_output\NIFTY 2022-04-13 SRE_DAY_LOW_PSL No-1.parquet


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  5.99it/s]

KeyboardInterrupt

